# Diagnostic des maladies du maïs, entraînement dans Google Colab

Ce notebook entraîne un petit modèle de reconnaissance d'images qui fonctionne ensuite **sans connexion** dans l'application Conseil Rotation. Il s'exécute dans Google Colab, avec une carte graphique, pour ne rien télécharger sur l'ordinateur de l'équipe.

**Avant de lancer**, choisissez Exécution, Modifier le type d'exécution, puis GPU T4. Lancez ensuite Exécution, Tout exécuter. Comptez une à deux heures, dont le téléchargement d'environ 17 Go d'images par Colab.

**Données**, toutes prises au champ avec des téléphones, sous licence CC0 1.0 sur Harvard Dataverse, citées avec attribution comme le demandent leurs fiches techniques.

| Rôle | Jeu | Pays |
|---|---|---|
| Entraînement | Makerere University Maize Image Dataset, doi 10.7910/DVN/LPGHKK | Ouganda |
| Entraînement | NMAIST Maize dataset, doi 10.7910/DVN/GDON8Q | Tanzanie |
| Entraînement | NUST Maize Dataset, doi 10.7910/DVN/6R78HR | Namibie selon le nom de l'université |
| Test indépendant | KaraAgro AI Maize Dataset, doi 10.7910/DVN/CXUMDS | Ghana |

**Règles de rigueur**

- Le Ghana n'est jamais vu pendant l'entraînement. Il mesure la fiabilité sur un pays inconnu du modèle.
- L'helminthosporiose du nord n'existe que dans le jeu de l'Ouganda. Son test utilise une autre archive de ce même jeu, et sa fiabilité est publiée avec la mention qu'elle n'est pas testée sur une source indépendante.
- Le seuil de confiance en dessous duquel le modèle répond « inconnu » est choisi sur la validation, jamais sur le test.
- Des maladies et ravageurs hors de la liste, chenille légionnaire du Ghana et nécrose létale de Tanzanie, vérifient que le modèle ne les prend pas pour une maladie connue.
- Le modèle ne donne jamais de traitement. Il propose une maladie possible, à faire confirmer par un conseiller ANADER.

In [ ]:
!nvidia-smi -L || echo "Pas de carte graphique, choisissez GPU T4 dans le type d'exécution"
!apt-get -qq install -y unrar p7zip-full > /dev/null
!pip -q install onnx onnxruntime onnxscript

In [ ]:
import hashlib, json, math, os, random, shutil, subprocess, time
from collections import Counter, defaultdict
from pathlib import Path

import numpy as np
import requests
import torch
from PIL import Image

GRAINE = 20261114
random.seed(GRAINE); np.random.seed(GRAINE); torch.manual_seed(GRAINE)
RACINE = Path("/content/mais")
CLASSES = ["saine", "striure", "helminthosporiose_nord"]
MAX_PAR_ARCHIVE_ENTRAINEMENT = 1500
MAX_PAR_ARCHIVE_TEST = 1000
TAILLE = 224

# Rôle, jeu, nom exact du fichier sur Dataverse, classe ou étiquette hors liste.
DOI = {"MAKERERE": "10.7910/DVN/LPGHKK", "NMAIST": "10.7910/DVN/GDON8Q",
       "NUST": "10.7910/DVN/6R78HR", "KARAAGRO": "10.7910/DVN/CXUMDS"}
ARCHIVES = [
    ("entrainement", "NMAIST", "HEATHLY.zip", "saine"),
    ("entrainement", "NUST", "healthy.rar", "saine"),
    ("entrainement", "MAKERERE", "Healthy_1.rar", "saine"),
    ("entrainement", "NMAIST", "MSV_1.zip", "striure"),
    ("entrainement", "NUST", "MSV.rar", "striure"),
    ("entrainement", "MAKERERE", "MSV_1.rar", "striure"),
    ("entrainement", "MAKERERE", "MLB_1.rar", "helminthosporiose_nord"),
    ("test_independant", "KARAAGRO", "healthy_01.rar", "saine"),
    ("test_independant", "KARAAGRO", "maize_streak_01.rar", "striure"),
    ("test_meme_source", "MAKERERE", "MLB_2.rar", "helminthosporiose_nord"),
    ("hors_liste", "KARAAGRO", "faw_05.rar", "chenille_legionnaire"),
    ("hors_liste", "NMAIST", "MLN.zip", "necrose_letale"),
]

### 1. Téléchargement vérifié

Chaque archive est téléchargée depuis l'API officielle de Harvard Dataverse, vérifiée par son empreinte MD5 publiée, extraite, puis supprimée pour libérer le disque.

In [ ]:
def fichiers_dataverse(doi):
    url = f"https://dataverse.harvard.edu/api/datasets/:persistentId/?persistentId=doi:{doi}"
    version = requests.get(url, timeout=60).json()["data"]["latestVersion"]
    return {f["dataFile"]["filename"]: f["dataFile"] for f in version["files"]}, version.get("license", {}).get("name")

def md5(chemin):
    h = hashlib.md5()
    with open(chemin, "rb") as f:
        for bloc in iter(lambda: f.read(1 << 22), b""):
            h.update(bloc)
    return h.hexdigest()

def extraire(archive, dossier):
    dossier.mkdir(parents=True, exist_ok=True)
    commande = ["unrar", "x", "-o+", "-inul", str(archive), str(dossier) + "/"] if archive.suffix == ".rar" \
        else ["7z", "x", "-y", "-bd", f"-o{dossier}", str(archive)]
    subprocess.run(commande, check=True, stdout=subprocess.DEVNULL)

catalogue, licences = {}, {}
for jeu, doi in DOI.items():
    catalogue[jeu], licences[jeu] = fichiers_dataverse(doi)
print("Licences lues sur Dataverse", licences)

journal = []
for role, jeu, nom, etiquette in ARCHIVES:
    dossier = RACINE / role / jeu / Path(nom).stem
    if dossier.exists() and any(dossier.rglob("*")):
        continue
    info = catalogue[jeu][nom]
    archive = RACINE / "archives" / nom
    archive.parent.mkdir(parents=True, exist_ok=True)
    debut = time.time()
    with requests.get(f"https://dataverse.harvard.edu/api/access/datafile/{info['id']}", stream=True, timeout=600) as r:
        r.raise_for_status()
        with open(archive, "wb") as f:
            for bloc in r.iter_content(1 << 22):
                f.write(bloc)
    attendu = info.get("md5") or info.get("checksum", {}).get("value")
    if md5(archive) != attendu:
        raise ValueError(f"empreinte MD5 différente pour {nom}")
    extraire(archive, dossier)
    archive.unlink()
    journal.append((nom, round(info["filesize"] / 1e9, 2), round(time.time() - debut)))
    print(f"{nom} vérifié et extrait, {journal[-1][1]} Go en {journal[-1][2]} s")

### 2. Inventaire et tirage des images

Un nombre maximal d'images est tiré au hasard dans chaque archive, avec une graine fixe, pour que l'entraînement reste court et reproductible.

In [ ]:
EXTENSIONS = {".jpg", ".jpeg", ".png"}

def images(dossier):
    return sorted(p for p in dossier.rglob("*") if p.suffix.lower() in EXTENSIONS)

def lisible(p):
    try:
        with Image.open(p) as im:
            im.verify()
        return True
    except Exception:
        return False

ensembles = defaultdict(list)
inventaire = []
for role, jeu, nom, etiquette in ARCHIVES:
    trouvees = images(RACINE / role / jeu / Path(nom).stem)
    maximum = MAX_PAR_ARCHIVE_ENTRAINEMENT if role == "entrainement" else MAX_PAR_ARCHIVE_TEST
    tirees = random.Random(GRAINE).sample(trouvees, min(maximum, len(trouvees)))
    tirees = [p for p in tirees if lisible(p)]
    ensembles[role] += [(p, etiquette, jeu) for p in tirees]
    inventaire.append((role, jeu, nom, etiquette, len(trouvees), len(tirees)))
for ligne in inventaire:
    print(*ligne)

### 3. Validation tirée des seules sources d'entraînement

In [ ]:
entrainement = ensembles["entrainement"]
random.Random(GRAINE).shuffle(entrainement)
par_classe = defaultdict(list)
for e in entrainement:
    par_classe[e[1]].append(e)
apprentissage, validation = [], []
for classe, liste in par_classe.items():
    coupe = int(0.15 * len(liste))
    validation += liste[:coupe]
    apprentissage += liste[coupe:]
print("Apprentissage", Counter(e[1] for e in apprentissage))
print("Validation", Counter(e[1] for e in validation))

### 4. Modèle

MobileNetV3 Small, pré-entraîné sur ImageNet, choisi pour sa petite taille et sa vitesse sur un téléphone d'entrée de gamme. La tête est d'abord entraînée seule, puis tout le réseau est ajusté avec un taux d'apprentissage faible.

In [ ]:
import torch.nn as nn
from torch.utils.data import DataLoader, Dataset
from torchvision import models, transforms

MOYENNE, ECART = [0.485, 0.456, 0.406], [0.229, 0.224, 0.225]
appareil = "cuda" if torch.cuda.is_available() else "cpu"
transformation_apprentissage = transforms.Compose([
    transforms.RandomResizedCrop(TAILLE, scale=(0.6, 1.0)), transforms.RandomHorizontalFlip(),
    transforms.RandomVerticalFlip(), transforms.ColorJitter(0.3, 0.3, 0.2), transforms.ToTensor(),
    transforms.Normalize(MOYENNE, ECART)])
transformation_evaluation = transforms.Compose([
    transforms.Resize(256), transforms.CenterCrop(TAILLE), transforms.ToTensor(), transforms.Normalize(MOYENNE, ECART)])

class Feuilles(Dataset):
    def __init__(self, elements, transformation):
        self.elements, self.transformation = elements, transformation
    def __len__(self):
        return len(self.elements)
    def __getitem__(self, i):
        chemin, etiquette, _ = self.elements[i]
        image = Image.open(chemin).convert("RGB")
        cible = CLASSES.index(etiquette) if etiquette in CLASSES else -1
        return self.transformation(image), cible

charger = lambda e, t, melange: DataLoader(Feuilles(e, t), batch_size=64, shuffle=melange, num_workers=2,
                                          pin_memory=True)
modele = models.mobilenet_v3_small(weights=models.MobileNet_V3_Small_Weights.IMAGENET1K_V1)
modele.classifier[3] = nn.Linear(modele.classifier[3].in_features, len(CLASSES))
modele = modele.to(appareil)
poids = torch.tensor([1 / max(1, sum(e[1] == c for e in apprentissage)) for c in CLASSES], device=appareil)
critere = nn.CrossEntropyLoss(weight=poids / poids.sum() * len(CLASSES))

def epoques(nombre, parametres, taux):
    optimiseur = torch.optim.AdamW(parametres, lr=taux, weight_decay=1e-4)
    for epoque in range(nombre):
        modele.train()
        total = 0.0
        for x, y in charger(apprentissage, transformation_apprentissage, True):
            x, y = x.to(appareil), y.to(appareil)
            optimiseur.zero_grad()
            perte = critere(modele(x), y)
            perte.backward()
            optimiseur.step()
            total += perte.item() * len(y)
        print(f"époque {epoque + 1}, perte moyenne {total / len(apprentissage):.4f}")

for p in modele.features.parameters():
    p.requires_grad = False
epoques(3, modele.classifier.parameters(), 1e-3)
for p in modele.parameters():
    p.requires_grad = True
epoques(5, modele.parameters(), 1e-4)

### 5. Seuil de confiance, choisi sur la validation

Le seuil est le 5e centile de la confiance des images de validation correctement classées. Une photo dont la confiance est plus basse reçoit la réponse « inconnu ».

In [ ]:
@torch.no_grad()
def predire(elements):
    modele.eval()
    probabilites, cibles = [], []
    for x, y in charger(elements, transformation_evaluation, False):
        probabilites.append(torch.softmax(modele(x.to(appareil)), 1).cpu())
        cibles.append(y)
    return torch.cat(probabilites).numpy(), torch.cat(cibles).numpy()

p_val, y_val = predire(validation)
justes = p_val.argmax(1) == y_val
SEUIL = float(np.percentile(p_val.max(1)[justes], 5))
print(f"Exactitude en validation {justes.mean():.3f}, seuil retenu {SEUIL:.3f}")

### 6. Fiabilité sur les tests

Pour chaque classe, le rappel est la part des images reconnues avec la bonne classe et une confiance suffisante. La précision est la part des réponses d'une classe qui sont justes. Les intervalles à 95 % sont ceux de Wilson.

In [ ]:
def wilson(k, n, z=1.959964):
    if n == 0:
        return [None, None]
    p = k / n
    centre = (p + z * z / (2 * n)) / (1 + z * z / n)
    demi = z * math.sqrt(p * (1 - p) / n + z * z / (4 * n * n)) / (1 + z * z / n)
    return [round(max(0, centre - demi), 3), round(min(1, centre + demi), 3)]

def reponses(p):
    return np.where(p.max(1) >= SEUIL, p.argmax(1), -1)

fiabilite = {}
for role in ["test_independant", "test_meme_source"]:
    p, y = predire(ensembles[role])
    r = reponses(p)
    for i, classe in enumerate(CLASSES):
        n = int((y == i).sum())
        if not n:
            continue
        k = int(((r == i) & (y == i)).sum())
        rendues = int((r == i).sum())
        fiabilite[f"{role}_{classe}"] = {
            "images": n, "rappel": round(k / n, 3), "rappel_ic95": wilson(k, n),
            "precision": round(k / rendues, 3) if rendues else None,
            "precision_ic95": wilson(k, rendues) if rendues else None,
            "part_inconnu": round(float((r[y == i] == -1).mean()), 3)}
p, _ = predire(ensembles["hors_liste"])
r = reponses(p)
etiquettes = np.array([e[1] for e in ensembles["hors_liste"]])
for etiquette in sorted(set(etiquettes)):
    choisi = r[etiquettes == etiquette]
    fiabilite[f"hors_liste_{etiquette}"] = {
        "images": int(len(choisi)), "part_inconnu": round(float((choisi == -1).mean()), 3),
        "part_saine": round(float((choisi == 0).mean()), 3),
        "part_maladie_connue": round(float((choisi > 0).mean()), 3)}
print(json.dumps(fiabilite, indent=1, ensure_ascii=False))

### 7. Export pour le téléphone

Le modèle est exporté au format ONNX, vérifié contre PyTorch sur une image réelle, puis rangé avec sa fiche dans une archive à télécharger. Placez ensuite les deux fichiers dans `app/diagnostic/` du dépôt.

In [ ]:
import onnxruntime as ort
modele = modele.cpu().eval()
SORTIE = Path("/content/diagnostic")
SORTIE.mkdir(exist_ok=True)
exemple = torch.randn(1, 3, TAILLE, TAILLE)
try:
    torch.onnx.export(modele, exemple, SORTIE / "modele.onnx", input_names=["image"], output_names=["logits"],
                      opset_version=17, dynamo=False)
except TypeError:
    torch.onnx.export(modele, exemple, SORTIE / "modele.onnx", input_names=["image"], output_names=["logits"],
                      opset_version=17)
image = transformation_evaluation(Image.open(ensembles["test_independant"][0][0]).convert("RGB")).unsqueeze(0)
ecart_max = float(np.abs(ort.InferenceSession(str(SORTIE / "modele.onnx")).run(None, {"image": image.numpy()})[0]
                         - modele(image).detach().numpy()).max())
print("Écart maximal entre ONNX et PyTorch", ecart_max)
assert ecart_max < 1e-3

fiche = {
    "classes": CLASSES, "taille": TAILLE, "redimension": 256, "moyenne": MOYENNE, "ecart_type": ECART,
    "seuil": round(SEUIL, 4), "fiabilite": fiabilite, "inventaire": inventaire, "licences": licences,
    "sources": DOI, "graine": GRAINE, "torch": torch.__version__, "date": time.strftime("%Y-%m-%d"),
    "avertissement": "Maladie possible, à faire confirmer par votre conseiller ANADER. Aucun traitement n'est proposé.",
}
(SORTIE / "modele.json").write_text(json.dumps(fiche, indent=1, ensure_ascii=False), encoding="utf-8")
print("Taille du modèle", round((SORTIE / "modele.onnx").stat().st_size / 1e6, 1), "Mo")
shutil.make_archive("/content/diagnostic", "zip", SORTIE)
from google.colab import files
files.download("/content/diagnostic.zip")